In [ ]:
# ============ سلول 0: نصب Meep/MPB با conda ============
!pip install -q condacolab
import condacolab
condacolab.install()   # بعد از اجرا، runtime خودکار ریستارت می‌شود؛ سلول بعدی را اجرا کنید

In [ ]:
# بعد از ریستارت اجرا کنید (حدود ۳ تا ۵ دقیقه)
!mamba install -y -q -c conda-forge pymeep=*=nompi_* pandas matplotlib
import meep as mp
from meep import mpb
print("Meep version:", mp.__version__)

In [ ]:
# ============ سلول 1: راه‌اندازی ============
from google.colab import drive
drive.mount('/content/drive')

import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
import meep as mp
from meep import mpb

OUT = "/content/drive/MyDrive/PIC_Project1_Meep"
os.makedirs(OUT, exist_ok=True)

H, W0 = 0.22, 0.45      # ضخامت و عرض نامی (µm)؛ a = 1 µm واحد طول Meep
SY, SZ = 3.0, 3.0       # ابعاد سلول مقطع (µm)
RES = 64                # پیکسل بر µm (برای حساسیت ساخت بالاتر می‌بریم)
print("Output:", OUT)

In [ ]:
# ============ سلول 2: مواد و تابع n_eff ============
def n_si(wl):
    """Si: Sellmeier. wl بر حسب µm؛ در 1.55 حدود 3.476"""
    l2 = wl**2
    return np.sqrt(1 + 10.6684293*l2/(l2-0.301516485**2)
                     + 0.003043475*l2/(l2-1.13475115**2)
                     + 1.54133408*l2/(l2-1104.0**2))

def n_sio2(wl):
    """SiO2: Malitson. در 1.55 حدود 1.444"""
    l2 = wl**2
    return np.sqrt(1 + 0.6961663*l2/(l2-0.0684043**2)
                     + 0.4079426*l2/(l2-0.1162414**2)
                     + 0.8974794*l2/(l2-9.896161**2))

# پاریتی‌ها (موج‌بر در راستای x، y افقی، z عمودی)
PAR = {"TE": mp.EVEN_Z + mp.ODD_Y,     # TE0
       "TM": mp.ODD_Z  + mp.EVEN_Y}    # TM0

def make_solver(width, height, wl, res=RES, nb=1):
    """هسته Si در محیط SiO2 (BOX و کلدینگ هر دو اکسید). ضریب شکست‌ها در همین λ"""
    geom = [mp.Block(size=mp.Vector3(mp.inf, width, height),
                     center=mp.Vector3(),
                     material=mp.Medium(index=n_si(wl)))]
    return mpb.ModeSolver(num_bands=nb, geometry=geom,
                          geometry_lattice=mp.Lattice(size=mp.Vector3(0, SY, SZ)),
                          resolution=res, tolerance=1e-7,
                          default_material=mp.Medium(index=n_sio2(wl)))

def neff(width, height, wl, pol="TE", res=RES):
    """n_eff بالاترین مد با پاریتی مورد نظر. اگر هدایت‌شده نباشد NaN"""
    ms = make_solver(width, height, wl, res)
    om, n_ox = 1/wl, n_sio2(wl)
    try:
        k = ms.find_k(PAR[pol], om, 1, 1, mp.Vector3(1), 1e-6,
                      2.4*om, n_ox*om, n_si(wl)*om)[0]
    except Exception:
        return np.nan
    n = k/om                                  # n_eff = k * lambda
    return n if n > n_ox + 1e-3 else np.nan

print("n_Si(1.55)=%.4f  n_SiO2(1.55)=%.4f" % (n_si(1.55), n_sio2(1.55)))
print("TE0 n_eff =", neff(W0, H, 1.55, "TE"), "  (انتظار ~2.4)")
print("TM0 n_eff =", neff(W0, H, 1.55, "TM"), "  (انتظار ~1.7)")

# ---- پروفایل مد ----
def plot_mode(pol, comp, fname):
    wl = 1.55; om = 1/wl
    ms = make_solver(W0, H, wl)
    ms.k_points = [mp.Vector3(neff(W0, H, wl, pol)*om)]
    ms.run_parity(PAR[pol], True)
    E = np.squeeze(np.abs(ms.get_efield(1, False)))      # (ny, nz, 3)
    eps = np.squeeze(ms.get_epsilon())
    ext = [-SY/2, SY/2, -SZ/2, SZ/2]
    plt.figure(figsize=(4.5, 4))
    plt.imshow(E[:, :, comp].T**2, origin="lower", extent=ext, cmap="inferno")
    plt.contour(np.linspace(-SY/2, SY/2, eps.shape[0]),
                np.linspace(-SZ/2, SZ/2, eps.shape[1]), eps.T, levels=[6], colors="w")
    plt.xlim(-1, 1); plt.ylim(-1, 1)
    plt.title(f"{pol}0 |E_{'yz'[comp-1]}|²"); plt.xlabel("y (µm)"); plt.ylabel("z (µm)")
    plt.savefig(f"{OUT}/{fname}", dpi=200); plt.show()

plot_mode("TE", 1, "mode_TE0.png")   # مؤلفه Ey
plot_mode("TM", 2, "mode_TM0.png")   # مؤلفه Ez

In [ ]:
# ============ سلول 3: جاروب عرض/طول موج ============
widths = np.arange(0.30, 0.801, 0.05)
wls    = np.arange(1.50, 1.601, 0.02)
fn = f"{OUT}/sweep_neff.npz"

if os.path.exists(fn):
    d = np.load(fn); neff_te, neff_tm = d["te"], d["tm"]; print("Loaded from Drive.")
else:
    neff_te = np.full((len(widths), len(wls)), np.nan); neff_tm = neff_te.copy()
    for i, w in enumerate(widths):
        for j, wl in enumerate(wls):
            neff_te[i, j] = neff(w, H, wl, "TE")
            neff_tm[i, j] = neff(w, H, wl, "TM")
        print(f"width {w*1e3:.0f} nm done")
    np.savez(fn, te=neff_te, tm=neff_tm, widths=widths, wls=wls)

def group_index(row):
    ok = ~np.isnan(row)
    if ok.sum() < 4: return np.full_like(row, np.nan)
    p = np.polyfit(wls[ok], row[ok], 2)          # برازش مرتبه 2 برای مشتق
    return row - wls*np.polyval(np.polyder(p), wls)   # n_g = n_eff - λ dn/dλ

ng_te = np.array([group_index(r) for r in neff_te])
ng_tm = np.array([group_index(r) for r in neff_tm])

ic = np.argmin(abs(wls-1.55))
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].plot(widths*1e3, neff_te[:, ic], 'o-', label="TE0")
ax[0].plot(widths*1e3, neff_tm[:, ic], 's-', label="TM0")
ax[0].axhline(n_sio2(1.55), ls='--', c='gray', label="n_clad")
ax[0].set(xlabel="Width (nm)", ylabel="n_eff", title="n_eff @1550 nm"); ax[0].legend(); ax[0].grid()
for i in [0, 3, 6, 10]:
    ax[1].plot(wls*1e3, neff_te[i], label=f"w={widths[i]*1e3:.0f} nm")
ax[1].set(xlabel="Wavelength (nm)", ylabel="n_eff (TE0)", title="Dispersion"); ax[1].legend(); ax[1].grid()
ax[2].plot(widths*1e3, ng_te[:, ic], 'o-', label="TE0")
ax[2].plot(widths*1e3, ng_tm[:, ic], 's-', label="TM0")
ax[2].set(xlabel="Width (nm)", ylabel="n_g", title="Group index @1550 nm"); ax[2].legend(); ax[2].grid()
plt.tight_layout(); plt.savefig(f"{OUT}/neff_ng_plots.png", dpi=200); plt.show()

df = pd.DataFrame({"width_nm": widths*1e3, "neff_TE0": neff_te[:, ic], "neff_TM0": neff_tm[:, ic],
                   "ng_TE0": ng_te[:, ic], "ng_TM0": ng_tm[:, ic]})
df.to_csv(f"{OUT}/neff_ng_table_1550.csv", index=False); df.round(4)

In [ ]:
# ============ سلول 4: تحلیل تک‌مدی (روش خط نور) ============
wl0 = 1.55
wf = np.arange(0.30, 1.001, 0.025)
fn = f"{OUT}/single_mode.json"

def count_guided(width, pol_parity, nb=6):
    """در k = n_clad*omega: باندهای با بسامد < omega هدایت‌شده‌اند"""
    ms = make_solver(width, H, wl0, nb=nb)
    om = 1/wl0
    ms.k_points = [mp.Vector3(n_sio2(wl0)*om)]
    ms.run_parity(pol_parity, True)
    f = np.array(ms.all_freqs[0])
    return int((f < om*(1-1e-4)).sum()), f

if os.path.exists(fn):
    res = json.load(open(fn))
else:
    res = {"w": [], "nTE": [], "nTM": [], "f_TE2nd": []}
    for w in wf:
        nte, fte = count_guided(w, mp.EVEN_Z)     # همه مدهای TE-مانند
        ntm, _   = count_guided(w, mp.ODD_Z)      # همه مدهای TM-مانند
        res["w"].append(float(w)); res["nTE"].append(nte); res["nTM"].append(ntm)
        res["f_TE2nd"].append(float(fte[1]))      # بسامد باند دوم TE (TE1) روی خط نور
    json.dump(res, open(fn, "w"))

w_arr = np.array(res["w"])*1e3
plt.figure(figsize=(7, 4))
plt.step(w_arr, res["nTE"], where="mid", label="# TE modes")
plt.step(w_arr, res["nTM"], where="mid", label="# TM modes")
plt.xlabel("Width (nm)"); plt.ylabel("Guided modes"); plt.grid(); plt.legend()
plt.title("Single-mode condition @1550 nm (h=220 nm)")
plt.savefig(f"{OUT}/single_mode_condition.png", dpi=200); plt.show()

# عرض cutoff مد TE1: جایی که بسامد باند دوم روی خط نور برابر 1/λ می‌شود
g = np.array(res["f_TE2nd"]) - 1/wl0
idx = np.where(np.diff(np.sign(g)) != 0)[0]
for i in idx:
    wc = np.interp(0, [g[i+1], g[i]], [w_arr[i+1], w_arr[i]])
    print(f"TE1 cutoff width ≈ {wc:.0f} nm  (عرض‌های کمتر از این: تک‌مدی TE)")
for w, a, b in zip(w_arr, res["nTE"], res["nTM"]):
    print(f"w={w:.0f} nm  TE={a}  TM={b}")

In [ ]:
# ============ سلول 5 (v3): اتلاف خمش با حلقه Meep ============
NEFF_TE = neff(W0, H, 1.55, "TE")      # از MPB: 2.352
NG_TE   = 4.2                          # بعد از سلول 3/6 با ng واقعی جایگزین کنید
Q_RELIABLE = 1e5                       # بالاتر از این، Harminv قابل اعتماد نیست

def ring_Q(R, w=W0, h=H, res=30, T=800):
    n_core, n_ox = n_si(1.55), n_sio2(1.55)
    dpml = 1.0
    sr = R + w/2 + 1.0 + dpml
    sz = h + 2*1.2 + 2*dpml
    fcen, df = 1/1.55, 0.05
    m = int(round(2*np.pi*R*NEFF_TE*fcen))          # عدد آزیموتی از n_eff واقعی
    geom = [mp.Block(center=mp.Vector3(R, 0, 0), size=mp.Vector3(w, mp.inf, h),
                     material=mp.Medium(index=n_core))]
    sim = mp.Simulation(cell_size=mp.Vector3(sr, 0, sz), geometry=geom,
                        default_material=mp.Medium(index=n_ox),
                        boundary_layers=[mp.PML(dpml, direction=mp.R, side=mp.High),
                                         mp.PML(dpml, direction=mp.Z)],
                        # *** منبع Er برای TE0 (اصلاح باگ) ***
                        sources=[mp.Source(mp.GaussianSource(fcen, fwidth=df),
                                           mp.Er, mp.Vector3(R, 0, 0))],
                        resolution=res, dimensions=mp.CYLINDRICAL, m=m)
    hm = mp.Harminv(mp.Er, mp.Vector3(R+0.05, 0, 0), fcen, df)
    sim.run(mp.after_sources(hm), until_after_sources=T)
    print(f"  R={R}: m={m}, تعداد مدهای Harminv = {len(hm.modes)}")
    for x in hm.modes:
        print(f"     f={x.freq:.5f}  Q={x.Q:.3g}  |amp|={abs(x.amp):.2e}  err={abs(x.err):.1e}")
    # abs(err) چون در حالت استوانه‌ای مختلط است
    good = [x for x in hm.modes
            if x.Q > 0 and abs(x.err) < 1e-3 and abs(x.freq - fcen) < df]
    if not good:
        return None
    best = max(good, key=lambda x: abs(x.amp))
    return best.freq, best.Q, m, abs(best.amp)

radii = np.array([1.5, 2.0, 2.5, 3.0, 4.0, 5.0])
fn = f"{OUT}/bend_loss_meep.csv"
if os.path.exists(fn): os.remove(fn)     # نتایج قبلی غلط بودند؛ پاک شود
rows = []

for R in radii:
    r = ring_Q(R)
    if r is None:
        print(f"R={R}: مد معتبر پیدا نشد"); continue
    f, Q, m, amp = r; lam = 1/f
    alpha = 2*np.pi*NG_TE/(lam*Q)
    loss_cm = 4.343*alpha*1e4
    rows.append(dict(R_um=float(R), m=m, lambda_res_nm=lam*1e3, Q_rad=Q, amp=amp,
                     reliable=bool(Q < Q_RELIABLE),
                     loss_dB_per_cm=loss_cm,
                     loss_dB_per_90deg=loss_cm*(np.pi*R/2)*1e-4))
    print(rows[-1])
    pd.DataFrame(rows).to_csv(fn, index=False)

dfb = pd.DataFrame(rows)
if len(dfb):
    plt.figure(figsize=(6, 4))
    rel = dfb[dfb.reliable]
    plt.semilogy(rel.R_um, rel.loss_dB_per_cm, 'o-', label="reliable")
    unr = dfb[~dfb.reliable]
    plt.semilogy(unr.R_um, unr.loss_dB_per_cm, 'v', c='r', label="Q too high (upper bound)")
    plt.xlabel("Bend radius (µm)"); plt.ylabel("Bend loss (dB/cm)")
    plt.grid(which="both"); plt.legend()
    plt.savefig(f"{OUT}/bend_loss.png", dpi=200); plt.show()
    print(dfb.round(6))

In [ ]:
# ============ سلول 6: حساسیت ±10 nm + همگرایی ============
d = 0.010
cases = {"nominal": (W0, H), "w+10": (W0+d, H), "w-10": (W0-d, H),
         "h+10": (W0, H+d), "h-10": (W0, H-d)}
RES_S = 128                       # رزولوشن بالا چون تغییر ۱۰ nm کوچک است
fn = f"{OUT}/fab_sensitivity.csv"

if os.path.exists(fn):
    dfs = pd.read_csv(fn)
else:
    rows = []
    wl3 = np.array([1.53, 1.55, 1.57])
    for name, (w, h) in cases.items():
        ne = np.array([neff(w, h, wl, "TE", res=RES_S) for wl in wl3])
        p = np.polyfit(wl3, ne, 2)
        ng = ne[1] - 1.55*np.polyval(np.polyder(p), 1.55)
        rows.append(dict(case=name, width_nm=w*1e3, height_nm=h*1e3, neff=ne[1], ng=ng))
        print(rows[-1])
    dfs = pd.DataFrame(rows); dfs.to_csv(fn, index=False)

g = lambda c, col: dfs[dfs.case == c][col].values[0]
nom_ng = g("nominal", "ng")
dn_dw = (g("w+10", "neff") - g("w-10", "neff"))/20.0     # بر nm
dn_dh = (g("h+10", "neff") - g("h-10", "neff"))/20.0
print(f"dn_eff/dw = {dn_dw:.2e} /nm   dn_eff/dh = {dn_dh:.2e} /nm")
print(f"Resonance shift: {1550*dn_dw/nom_ng:.3f} nm per nm width error")
print(f"Resonance shift: {1550*dn_dh/nom_ng:.3f} nm per nm height error")

# ---- تست همگرایی نسبت به رزولوشن ----
conv = pd.DataFrame([{"res": r, "neff_TE0": neff(W0, H, 1.55, "TE", res=r)} for r in [32, 48, 64, 96, 128]])
conv.to_csv(f"{OUT}/convergence.csv", index=False); print(conv)
dfs.round(4)